# BassSpecMatchPRO — NAM Colab
Worker público v3. O timbre vem somente da Referência; o app envia apenas o par **input.wav + output.wav** já produzido localmente.


In [ ]:
# BassSpecMatchPRO public worker — Colab/GPU, protocol v3
import os, sys, subprocess, secrets, json, time, threading, zipfile, shutil, re, urllib.request, base64, wave, hashlib
from pathlib import Path

WORKER_PROTOCOL = 'bassspec-nam-colab-v3'
WORKER_BUILD = 'bsm-colab-v3-20260821-b'
MAX_JOB_BYTES = 128 * 1024 * 1024

# Reexecução da célula encerra o endpoint e trainer anteriores desta mesma sessão.
_old_proc = globals().get('_BASSSPEC_ACTIVE_PROC')
if _old_proc is not None and _old_proc.poll() is None:
    _old_proc.terminate()
_old_server = globals().get('_BASSSPEC_SERVER')
if _old_server is not None:
    try: _old_server.shutdown()
    except Exception: pass
_old_tunnel = globals().get('_BASSSPEC_TUNNEL')
if _old_tunnel is not None and _old_tunnel.poll() is None:
    _old_tunnel.terminate()
_old_tunnel_log = globals().get('_BASSSPEC_TUNNEL_LOG')
if _old_tunnel_log is not None:
    try: _old_tunnel_log.close()
    except Exception: pass

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'flask', 'neural-amp-modeler==0.13.0'])
import torch
from flask import Flask, request, jsonify
from werkzeug.serving import make_server

CLOUDFLARED='/content/cloudflared'
if not (os.path.isfile(CLOUDFLARED) and os.access(CLOUDFLARED, os.X_OK)):
    tmp=CLOUDFLARED+'.download-'+secrets.token_hex(4)
    try:
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', tmp)
        os.chmod(tmp, 0o755)
        os.replace(tmp, CLOUDFLARED)
    finally:
        if os.path.exists(tmp): os.remove(tmp)

ROOT=Path('/content/bassspec_nam_worker')
ROOT.mkdir(parents=True, exist_ok=True)
TOKEN=secrets.token_urlsafe(32)
state_lock=threading.Lock()
state={'state':'ready','phase':'waiting','epoch':0,'epochs':0,'error':'','command':'','job_id':'','job_sha256':'','updated_at':time.time()}
app=Flask(__name__)

def auth():
    return request.headers.get('Authorization','') == 'Bearer '+TOKEN

def snapshot():
    with state_lock: return dict(state)

def update_state(**kwargs):
    kwargs['updated_at']=time.time()
    with state_lock: state.update(kwargs)

def gpu_info():
    available=bool(torch.cuda.is_available())
    device=torch.cuda.get_device_name(0) if available else 'cpu'
    return available, device

def validate_job(work):
    manifest=json.loads((work/'manifest.json').read_text())
    if manifest.get('format')!='bassspec-nam-colab-job-v3' or manifest.get('schema_version')!=3:
        raise ValueError('job/schema Colab v3 inválido')
    if manifest.get('training_mode')!='input-output':
        raise ValueError('training_mode deve ser input-output')
    if manifest.get('input_file')!='input.wav' or manifest.get('output_file')!='output.wav':
        raise ValueError('manifest input/output inválido')
    if manifest.get('reference_timbre_only') is not True or manifest.get('contains_source_audio') is not False:
        raise ValueError('proveniência do job inválida')
    if manifest.get('contains_reference_audio') is not False or (work/'reference.wav').exists():
        raise ValueError('reference.wav bruto não deve ser enviado ao worker')
    meta=[]
    payloads=[]
    for name in ('input.wav','output.wav'):
        path=work/name
        if not path.is_file():
            raise FileNotFoundError(name+' ausente; fallback --reference proibido')
        with wave.open(str(path),'rb') as w:
            info=(w.getframerate(),w.getnchannels(),w.getnframes())
            if info[0]!=manifest.get('sample_rate') or info[1]!=manifest.get('channels') or info[2]<=0:
                raise ValueError(name+' incompatível com manifest')
            frames=w.readframes(info[2])
            if not frames or not any(frames):
                raise ValueError(name+' silencioso')
            meta.append(info); payloads.append(frames)
    if meta[0]!=meta[1]:
        raise ValueError('input/output possuem formato ou comprimento incompatível')
    if payloads[0]==payloads[1]:
        raise ValueError('output.wav não difere de input.wav')
    return manifest

def safe_extract(zip_path, dest):
    dest=dest.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for member in z.infolist():
            target=(dest/member.filename).resolve()
            if target!=dest and dest not in target.parents:
                raise ValueError('ZIP contém caminho inválido')
        z.extractall(dest)

@app.get('/health')
def health():
    if not auth(): return ('Unauthorized',401)
    gpu,device=gpu_info()
    return jsonify({'ok':True,'protocol':WORKER_PROTOCOL,'worker_build':WORKER_BUILD,'gpu':gpu,'device':device})

@app.post('/job')
def job():
    if not auth(): return ('Unauthorized',401)
    gpu,device=gpu_info()
    if not gpu:
        return jsonify({'ok':False,'error':'GPU CUDA não está ativa no Colab','protocol':WORKER_PROTOCOL,'worker_build':WORKER_BUILD}),503
    job_id=request.args.get('job_id','').strip()
    try: epochs=int(request.args.get('epochs','25'))
    except Exception: epochs=25
    epochs=100 if epochs>=100 else 25
    if len(job_id)<16 or len(job_id)>80:
        return jsonify({'ok':False,'error':'job_id inválido'}),400
    current=snapshot()
    if current['state'] in ('received','training'):
        return jsonify({'ok':False,'error':'worker ocupado com outro job','job_id':current['job_id']}),409
    try:
        raw=request.get_data(cache=False)
        if not raw: raise ValueError('job ZIP vazio')
        if len(raw)>MAX_JOB_BYTES: raise ValueError('job ZIP excede 128 MiB')
        job_root=ROOT/'jobs'/job_id
        shutil.rmtree(job_root,ignore_errors=True); job_root.mkdir(parents=True)
        z=job_root/'job.zip'; z.write_bytes(raw)
        work=job_root/'job'; work.mkdir()
        safe_extract(z,work)
        validate_job(work)
        digest=hashlib.sha256(raw).hexdigest()
    except Exception as e:
        update_state(state='error',phase='rejected',error=str(e),command='',job_id=job_id,job_sha256='')
        return jsonify({'ok':False,'error':str(e),'job_id':job_id}),400
    update_state(state='received',phase='queued',epoch=0,epochs=epochs,error='',command='',job_id=job_id,job_sha256=digest)
    return jsonify({'ok':True,'state':'received','epochs':epochs,'job_id':job_id,'job_sha256':digest,'protocol':WORKER_PROTOCOL,'worker_build':WORKER_BUILD,'gpu':True,'device':device})

@app.get('/status')
def status():
    if not auth(): return ('Unauthorized',401)
    requested=request.args.get('job_id','').strip(); current=snapshot()
    if not requested or requested!=current.get('job_id',''):
        return jsonify({'error':'job_id não corresponde ao worker atual','job_id':current.get('job_id','')}),409
    current['protocol']=WORKER_PROTOCOL; current['worker_build']=WORKER_BUILD
    return jsonify(current)

@app.get('/result')
def result():
    if not auth(): return ('Unauthorized',401)
    requested=request.args.get('job_id','').strip(); current=snapshot()
    if not requested or requested!=current.get('job_id',''):
        return jsonify({'error':'job_id não corresponde ao resultado atual','job_id':current.get('job_id','')}),409
    p=ROOT/'jobs'/requested/'result'/'trained.nam'
    if current['state']!='done' or not p.exists(): return jsonify({'error':'not ready','job_id':requested}),409
    return jsonify({'ok':True,'job_id':requested,'protocol':WORKER_PROTOCOL,'worker_build':WORKER_BUILD,'nam_b64':base64.b64encode(p.read_bytes()).decode()})

def train_loop():
    global _BASSSPEC_ACTIVE_PROC
    while True:
        current=snapshot()
        if current['state']=='received':
            job_id=current['job_id']
            try:
                job_root=ROOT/'jobs'/job_id; work=job_root/'job'; validate_job(work)
                epochs=int(current.get('epochs',25)); out=job_root/'result'; shutil.rmtree(out,ignore_errors=True); out.mkdir()
                cmd=[sys.executable,str(work/'train_nam_official.py'),'--input',str(work/'input.wav'),'--output',str(work/'output.wav'),'--outdir',str(out),'--template',str(work/'model.nam'),'--epochs',str(epochs),'--ignore-checks']
                update_state(state='training',phase='starting',command=' '.join(cmd),error='',job_id=job_id)
                log_path=out/'trainer.log'
                with log_path.open('w') as log_file:
                    _BASSSPEC_ACTIVE_PROC=subprocess.Popen(cmd,cwd=work,stdout=log_file,stderr=subprocess.STDOUT,text=True)
                    progress=out/'training-progress.json'
                    while _BASSSPEC_ACTIVE_PROC.poll() is None:
                        phase='training'; epoch=snapshot().get('epoch',0)
                        if progress.exists():
                            try:
                                d=json.loads(progress.read_text()); phase=d.get('phase','training'); epoch=int(d.get('epoch',epoch))
                            except Exception: pass
                        update_state(state='training',phase=phase,epoch=epoch,job_id=job_id)
                        time.sleep(1)
                    rc=_BASSSPEC_ACTIVE_PROC.returncode
                if rc:
                    log=log_path.read_text(errors='replace')[-4000:] if log_path.exists() else ''
                    raise RuntimeError(log or ('trainer failed: '+str(rc)))
                model=out/'trained.nam'; data=json.loads(model.read_text())
                if data.get('architecture')!='SlimmableContainer': raise ValueError('trained.nam possui arquitetura inesperada')
                update_state(state='done',phase='complete',epoch=epochs,epochs=epochs,job_id=job_id)
            except Exception as e:
                update_state(state='error',phase='failed',error=str(e),job_id=job_id)
        time.sleep(.5)

threading.Thread(target=train_loop,daemon=True).start()
_server=make_server('127.0.0.1',0,app); _BASSSPEC_SERVER=_server; PORT=_server.server_port
threading.Thread(target=_server.serve_forever,daemon=True).start()

cloudflare_log=ROOT/'cloudflared.log'
try: cloudflare_log.unlink()
except FileNotFoundError: pass
_BASSSPEC_TUNNEL_LOG=open(cloudflare_log,'w')
tunnel=subprocess.Popen([CLOUDFLARED,'tunnel','--url','http://127.0.0.1:'+str(PORT),'--no-autoupdate','--loglevel','info'],stdout=_BASSSPEC_TUNNEL_LOG,stderr=subprocess.STDOUT,text=True)
_BASSSPEC_TUNNEL=tunnel
public=None; deadline=time.time()+45
while time.time()<deadline:
    if tunnel.poll() is not None: break
    if cloudflare_log.exists():
        log_text=cloudflare_log.read_text(errors='replace')
        m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',log_text)
        if m: public=m.group(0); break
    time.sleep(.25)
if not public:
    tunnel.terminate(); _BASSSPEC_TUNNEL_LOG.close(); raise RuntimeError('Cloudflare Quick Tunnel não iniciou; execute a célula novamente.')

healthy=False
for _ in range(20):
    try:
        req=urllib.request.Request(public+'/health',headers={'Authorization':'Bearer '+TOKEN})
        with urllib.request.urlopen(req,timeout=5) as resp:
            d=json.loads(resp.read().decode())
            if d.get('ok') and d.get('worker_build')==WORKER_BUILD: healthy=True; break
    except Exception: time.sleep(.5)
if not healthy:
    tunnel.terminate(); raise RuntimeError('Tunnel publicado, mas /health não ficou acessível.')

gpu,device=gpu_info()
print('BASSSPEC_COLAB='+public+'|'+TOKEN, flush=True)
print('Worker v3 pronto: build='+WORKER_BUILD+' | GPU='+device+' | upload ZIP bruto | treino obrigatório --input/--output | tunnel_pid='+str(tunnel.pid)+'.', flush=True)
